# Exploring and visualising data

<a id='Contents'></a>
## Contents
In this notebook we will practice exploring a dataset using data from Californian housing to try to understand what affects house prices. We will:

- [Import](#import) some data 
- [Prepare](#prepare) the data so we can explore it
- [Explore](#explore) the data
    - Describing the data
    - Visualising the data
    - Skewness
    - Comparing variables
    - Comparing everything in one go
    - Relationships in the data
- [Reshaping](#reshape) the data

## How to use this notebook

In this notebook all the code needed for the analysis has been written. You will be asked to reflect on the outputs of the code. Those tasks will be in one of two types of boxes:

<div class="alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for
</div>

<div class="alert-success">
<b>Question boxes:</b> <br>
These have questions you should answer before progressing.
</div>

In both cases, if you wish to add your answers to this notebook as you progress, add an extra `markdown` cell to contain your answers. 

If you wish to save your answers at the end of the workshop, remember to download the notebook to your computer.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
You don't need to understand all the code here for now, just look for:
<ul>
<li> What we are trying to do with each code cell. How does it fit in our objective?
<li> What the outputs of each code cell tell us? Are we reading too much into the results of each code cell?
</ul>
</div>

## The background to the data
We are going to explore data about [houses in California districts](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset), taken from the 1990 U.S. census by (Kelley Pace and Barry, 1997). Each row represents a census block group: the smallest geographical unit for which the U.S. Census Bureau publishes sample data. Each block group normally contains a population of 600 to 3,000 people.

Data Information:
- **MedHouseVal**:   median house value, expressed in hundreds of thousands of dollars (\$100,000). This is the value we will look to predict in the next workshop
- **MedInc**:        median income, expressed in tens of thousands of dollars (\$10,000) in the block group
- **HouseAge**:      median house age in the block group
- **AveRooms**:      average number of rooms per household in the block group
- **AveBedrms**:     average number of bedrooms per household in the block group
- **Population**:    the block group population
- **AveOccup**:      average number of household members in the block group
- **Latitude**:      block group latitude
- **Longitude**:     block group longitude


A household is a group of people residing within a home. Since the average number of rooms and bedrooms in this dataset are provided per household, these columns may take surprisingly large values for block groups with few households and many empty houses, such as vacation resorts.

---
<a id="import"></a>
## Import packages and read data
[Back to Contents](#Contents)

Let's start by importing the Python packages we will need:
- [**pandas**](https://pandas.pydata.org/): a tabular data manipulation package
- [**matplotlib**](https://matplotlib.org/): a data visualisation package
- [**seaborn**](https://seaborn.pydata.org/): a data visualisation package built on matplotlib that works well with pandas
- [**scipy**](https://docs.scipy.org/): a scientific function package
- [**numpy**](https://numpy.org/): a numerical analysis package
- [**sklearn**](https://scikit-learn.org/stable/): a common machine learning package

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import power_transform

sns.set_style("whitegrid") 

We can now import the data using the function [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) and have a look at the first few rows using the method [`head`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)

In [ ]:
independent_data, dependent_data = fetch_california_housing(return_X_y=True, as_frame=True)
data = independent_data.merge(dependent_data, left_index=True, right_index=True)
data.head()

---
<a id="prepare"></a>
## Prepare the data
[Back to Contents](#Contents)

For this notebook we are assuming the data has been prepared before being loaded in. However, it is always important to check that you have what you expect.

We can look at what kind of data our table contains using the [Pandas `info`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) method. 

<div class="alert-info">
<b>We should be asking ourselves:</b> <br>

- Does the data contain the columns I expect?
- Do the columns have the data type I would expect?
- Do any of the columns have missing values? Are these in any columns we intend to use? There cannot be any null values in the rows we plan to use in our model.

If you wish to add your answers to this notebook as you progress, add an extra `markdown` cell below this one with your answers. 
</div>

In [ ]:
data.info()

None of the columns have null (missing) values, so we can start exploring the data. There is also no column designed to only be an index.

---
<a id="explore"></a>
## Exploring the data
[Back to Contents](#Contents)

### Describing the data

Now that we have checked our data is clean, we can explore it. A good starting point is to look at the ***descriptive statistics*** and check that they seem reasonable. 

<div class="alert-info">
<b>We should be asking ourselves:</b> <br>

- Does the data have unexpected outliers (looking at the max and min values) that might suggest a data quality issue?
- Is the spread of the data what you would expect?
</div>

We can do so using the [`describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) method. It will give us the:
- **count**: number of non-null values in each column.
- **mean**: average value of each column.
- **std**: standard deviation of each column. Standard deviation measures the average spread from the mean value.
- **min**: minimum value of each column. When the values are placed in order, this will be the smallest and the first.
- **25%**: the value 25% of the way through the data when ordered. Also known as the value at the first quartile of the distribution.
- **median**: the value 50% of the way through the data when ordered. Also known as the value at the second quartile of the distribution.
- **75%**: the value 75% of the way through the data when ordered. Also known as the value at the third quartile of the distribution.
- **max**: maximum value of each column. When the values are placed in order, this will be the largest and the last.

In [ ]:
data.describe()

<div class="alert-success">
<b>Question:</b> <br>
What do you see in the descriptive statistics? 
<ul>
<li> Are there any columns that seem to have unexpected values?
<li> What would be the lowest value you would expect for each column?
<li> How close would you expect the mean and median to be? Which will be bigger?
<li> How close would you expect the median and max values to be?
</ul>
</div>

<a id="visualise"></a>
### Visualising the data

Getting a feel for the data from the descriptive statistics is hard. You can often get a better understanding by visualising the data. When visualising continuous data, histograms are a good place to start.

**Histograms** help us see the ***distribution*** of the data. A histogram splits the data into ***bins*** and counts how many values in the data fall into each bin. A histogram therefore shows us the frequency of different values in the data. To visualise a histogram it is typical to have the possible values of the data on the horizontal axis and the frequency of each value on the vertical axis.

We can create a histogram using the Seaborn [`histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html) function

In [ ]:
sns.histplot(data=data, x='MedHouseVal', bins=50, kde=True);

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Edit the code above to change the column name and see the histogram for different columns</p>
    <ul>
        <li> Are there any columns with two peaks (bimodal distribution)?
        <li> What would be the lowest value you would expect for each column?
        <li> After looking at the descriptive statistics, does anything about the distributions surprise you?
        <li> Try changing the number of bins in the histogram. What do you notice?
        <li> How centered are the distributions? 
    </ul>
</div>

<a id="skewness"></a>
### Skewness

We can see that some of the distributions are ***skewed***. Skew describes how symmetrical the distribution of a variable is. A distribution with ***no skew*** is symmetrical (mirrored) around the mean. A distribution with a ***positive skew*** has more of the data to the right of the mean in a long ***right tail***. Many distributions in the real world can't have negative numbers and have a long right tail, such as income, house size or house prices.

<img src="../images/skewness.png" alt="Skewness" width="900" class="center"/>

We can quantify the amount of skew in our variable distributions using the SciPy package statistics function [`skew`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skew.html). The closer the value returned is to zero the less skew there is and the more symmetrical the distribution is. Positive numbers signify that most of the data is to the right of the modal value, negative numbers to the left. `skew` is applied to each column individually using the Pandas method [`apply`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html).

In [ ]:
data.apply(sp.stats.skew)

How can this number help us to understand our distributions better?

Another way to visualise the distribution of the data is to use a ***violin plot***. A violin plot shows you the shape of the distribution of the data, the median, the interquartile range and the range of the data. We call it a violin plot because the histogram is mirrored in the main body of the plot, making it often look like a violin. We can create a violin plot using the Seaborn [`violinplot`](https://seaborn.pydata.org/generated/seaborn.violinplot.html) function.

In [ ]:
sns.violinplot(data=data, x='MedHouseVal');

### Comparing variables

We can compare the distributions of different variables by plotting them on the same graph. The most common way of doing this is to use a ***scatter plot***. A scatter plot shows a point for each data point in the data, with the horizontal position of the point representing the value of one variable and the vertical position representing the value of another variable. We can create a scatter plot using the Seaborn [`scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html) function.

In [ ]:
sns.scatterplot(data=data, x='MedInc', y='MedHouseVal');

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Edit the scatter plot code above to change the column names and see the relationships for different columns</p>
    <ul>
        <li> Are there any columns that seem related to each other?
        <li> Are there any that seem completely unrelated?
        <li> How good are scatter plots at showing the distribution of the data? Where are the densest areas?
        <li> How easy is it to spot outliers in scatter plots?
    </ul>
</div>

Scatter plots make it tricky to see the density of the data. One approach to visualising the density of the data is a heatmap. This can be done using the Seaborn [`heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) function. We will use that later, however, we can go further and visualise both the heatmap and plot the histograms of the data on the axes of the plot. We can do this using the Seaborn [`jointplot`](https://seaborn.pydata.org/generated/seaborn.jointplot.html) function. This function allows us to plot one graph in the middle and others on the sides. 

Another approach is to use colour to represent the density of the data. We can do this by replacing `scatter` with `hex` or `hist` as the `kind` parameter in the `jointplot` function.

In [ ]:
g = sns.jointplot(data=data, x='MedInc', y='MedHouseVal', kind='hist')
g.plot_marginals(sns.histplot, bins=50);

One `scatterplot` that might have caught your eye is the one between `Latitude` and `Longitude`. This is because these two columns represent the geographical location of the block group, which results in us seeing the shape of California. Why are the points not spread evenly across the state? What might this tell us about California?

In [ ]:
sns.scatterplot(data=data, x='Longitude', y='Latitude');

### Comparing everything in one go

Looking at one relationship at a time can be time-consuming and makes it hard to see the big picture. To look at every possible histogram and scatter plot at once, we can use another Seaborn function, [`pairplot`](https://seaborn.pydata.org/generated/seaborn.pairplot.html). This function will plot a grid of plots, with each variable plotted against every other variable. On the diagonal, where the two variables are the same, it will plot a histogram of the distribution of that variable.

<div class="alert-warning">
<b>Patience</b><br>
Pairplots take a long time to run. You will see that [*] appears to the left of the code box while it is running. Once it is complete you will see the star will be replaced by a number.
</div>

In [ ]:
sns.pairplot(data, corner=True);

There are many data visualisation we can use to explore the data. One of the best ways of seeing the different visualisations available is to look at collections of visualisations. Here are some examples for you to look through after the workshop:

- [Seaborn visualisation gallery](https://seaborn.pydata.org/examples/index.html)
- [Matplotlib visualisation gallery](https://matplotlib.org/stable/gallery/index.html)
- [Chart Doctor Visual Vocabulary](https://ft-interactive.github.io/visual-vocabulary/) and in the form of a [poster](https://github.com/Financial-Times/chart-doctor/blob/main/visual-vocabulary/Visual-vocabulary-en.pdf)
- [The Python Graph Gallery](https://python-graph-gallery.com/)
- [Office for National Statistics (ONS) Data Visualisation Guidance](https://service-manual.ons.gov.uk/data-visualisation/)
- A more general guidance is the online books: 
    - [Hands-On Data Visualization with Python](https://handsondataviz.org/)
    - [Fundamentals of Data Visualization](https://clauswilke.com/dataviz/)


<a id="relationships"></a>
## Relationships in the data

The plots we have looked at so far have shown us the relationships and shapes of the distributions of the data. We have seen that we can quantify the distributions using descriptive statistics and skewness. We can also quantify the relationships between variables using ***correlation***.

Correlations are a measure of how two variables are related to each other:
- A correlation between 0 and 1 means that as one variable increases, the other variable increases.
- A correlation between -1 and 0 means that as one variable increases, the other variable decreases.
- A correlation of 0 means that there is no relationship between the two variables.

<img src="../images/correlation.png" alt="Correlation" width="600" class="center"/>

We can calculate the correlation between two variables using the Pandas method [`corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html). This will return a table of the correlation between every pair of variables in the data, called a ***correlation matrix***.

In [ ]:
corr = data.corr()
corr

As before, we can see that just looking at the numbers in the correlation matrix can make it hard to spot patterns. We can use a ***heatmap*** to visualise the correlation matrix. A heatmap colours the cells of a grid based on the value in each cell. We can create a heatmap using the Seaborn [`heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) function. In this case we have cleaned it up to make it easier to read.

In [ ]:
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), 
            cmap='coolwarm', vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=True, fmt=".2f");

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Examine the heatmap above to understand the relationships between different columns</p>
    <ul>
        <li> Which columns are most related to each other?
        <li> Are there any that seem completely unrelated?
        <li> Which columns are most related to `MedHouseVal`? Keep in mind that a strong negative correlation is still a strong relationship.
        <li> Are there any correlations that seem weaker than you would have expected from the scatter plots?
    </ul>
</div>

<a id="reshape"></a>
## Reshaping the data

The value calculated for a correlation is affected by differences in skewness between the two variables. If one variable is very skewed, it can make the correlation seem weaker than it is. 

To correct for this, we can transform the data to make it less skewed and more symmetrical. One common transformation is the ***power transformation***. This transformation can help correct both positive and negative skewness. 

<div class="alert alert-block alert-warning">
<b>Remember</b><br>
We don't need to go in to the maths here. For this course you only need to know that transformations, including power transformation, exist and can help us to better understand the relationships in our data. If you want to understand what it will do to your data, the fastest way is to just try it and visualise it!
</div>

As we only want to transform the columns that have a large skew, we can create a list of all the columns with a large skew by filtering the list of columns based on those with an absolute skew greater than 1, i.e. those with a skew greater than 1 or less than -1.

In [ ]:
skewed_columns = data.columns[abs(data.apply(sp.stats.skew)) > 1]
skewed_columns

With this list of columns we can now apply the power transformation to the data. We can do this using the Scikit-learn function [`power_transform`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.power_transform.html).

In [ ]:
transformed_columns = pd.DataFrame(power_transform(data[skewed_columns]), columns=skewed_columns)

To help visualise the effect of the transformation, we can plot the violin plots of each column before and after the transformation.

In [ ]:
fig, axes = plt.subplots(len(skewed_columns), 1, figsize=(5, 20))
for ax, column in zip(axes, transformed_columns.columns):
    df = pd.concat([data[column], transformed_columns[column]], axis=1)
    df.columns = [column, f'{column}_transformed']
    sns.violinplot(data=df, inner='point', ax=ax);

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>What do you see that the power transform has done to each column?</p>
    <ul>
        <li> Where are the outliers?
        <li> Where are they centred?
        <li> Has the transform done the same thing for each column?
    </ul>
</div>

We can look at the transformed columns together using a pairplot

In [ ]:
sns.pairplot(transformed_columns, corner=True);

As these transformations have been helpful, we can put them back into the main dataset. To keep our analysis clean, we will make a copy first.

In [ ]:
data_transformed = data.copy()
data_transformed.loc[:, skewed_columns] = transformed_columns
data_transformed.head()

We can calculate the correlations between all the variables now we have transformed the data.

In [ ]:
corr_2 = data_transformed.corr()
sns.heatmap(corr_2, mask=np.triu(np.ones_like(corr, dtype=bool)), 
            cmap='coolwarm', vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=True, fmt=".2f");
corr_2

It can be hard to tell what has changed between the two correlation matrices. We can subtract one from the other to see the differences.

In [ ]:
sns.heatmap(corr-corr_2, mask=np.triu(np.ones_like(corr, dtype=bool)), 
            cmap='coolwarm', vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=True, fmt=".2f");
corr-corr_2

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Was the transformation helpful for understanding the data?</p>
    <ul>
        <li> What has changed in the correlations?
        <li> Are some correlations stronger than they were before?
        <li> How has the interpretability of the relationships changed?
    </ul>
</div>

We will use this reshaped data in the next workshop to build some models

# Bibliography

- Kelley Pace, R. and Barry, R. (1997) *Sparse spatial autoregressions, Statistics & Probability Letters*, 33(3), pp. 291–297. Available at: https://doi.org/10.1016/S0167-7152(96)00140-X and https://www.spatial-statistics.com/pace_manuscripts/spletters_ms_dir/statistics_prob_lets/pdf/fin_stat_letters.pdf
- Scikit-learn (2023) *California housing dataset*. Available at: https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset